# Get started with Kumo Relational

Kumo Relational is a foundation model for relational data: it predicts from linked tables, such as customers, products and orders, with no feature engineering and no model trained for each task.
You write what to predict as a short predictive query, and the model answers it from past examples in your own tables.

This notebook deploys the model in your workspace, then forecasts product demand, predicts which customers will stop ordering, predicts customer spend and large orders, fills in a product attribute, and forecasts a metric view measure.
Each kind of prediction is checked against what actually happened in Microsoft's AdventureWorks sample sales data.

## How Kumo Relational works

> **1. Enterprise data is a graph.**

Tables are connected by keys: here each order belongs to a customer, and each order line links a customer to a product.
These links give the model the context around every row, such as a customer's past orders.

> **2. Timestamps place events on a timeline.**

A table whose rows happen at a point in time, such as orders, is an event table.
You can pick any moment, the prediction time, and predict what happens next from what came before it.

> **3. Machine learning tasks can be written as predictive queries.**

A predictive query says what to predict, and for which entity, such as each product or each customer.
If you know SQL, the Predictive Query Language (PQL) will feel familiar:

```
PREDICT SUM(sales_order_details.OrderQty, 0, 30, days)   ← units ordered in the next 30 days
FOR EACH products.ProductID                              ← one prediction per product
```

There is no training step: for each request, the client reads past cases with known outcomes from your tables, called context examples, and the model predicts from them.

## Before you start

You need:

- the Kumo Relational listing installed from Databricks Marketplace;
- permission to create a Model Serving endpoint, in a region with GPU Model Serving;
- a catalog and schema where you can create tables;
- serverless compute with environment 4 or later, or Databricks Runtime 17.3 LTS or later.

The first deployment of the endpoint takes 20 to 40 minutes, then the rest of the notebook runs in a few minutes.
The endpoint bills for GPU time while it runs; it scales to zero when idle, and the last step shows how to delete it.

Run each cell in order with Shift+Enter.

**Let's get started!**

## Step 1. Install the client

Install the client, the `structured-data-models-client` package, with its Databricks extras, then restart Python.
pip may print a red message about `protobuf` versions; it does not affect this notebook.

In [ ]:
%pip install --quiet "structured-data-models-client[relational,databricks,databricks-serving]"
dbutils.library.restartPython()

## Step 2. Set your parameters

Set these names, then run the cell.
`MODEL_NAME` is the model from the listing, in the catalog you installed it into, and `CATALOG` and `SCHEMA` are where the sample tables go.

In [ ]:
import json
from datetime import timedelta

import pandas as pd
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
    ServingModelWorkloadType,
)
from sdm_client import (
    KumoRelational,
    PredictiveQuery,
    Relationship,
    Schema,
    SDMClient,
)
from sdm_connectors import DatabricksSource, quote_ident

# The model from the listing, as <catalog>.<schema>.<model>, and its version.
MODEL_NAME = 'kumo_relational.models.kumo_relational'
MODEL_VERSION = '1'
ENDPOINT_NAME = 'kumo-relational'
# Where the sample tables go: a catalog and schema you can create tables in.
CATALOG = 'main'
SCHEMA = 'kumo_relational_quickstart'


def quoted(*parts):
    # A Unity Catalog name, each part quoted for SQL.
    return '.'.join(quote_ident(part, char='`') for part in parts)


def full_name(table):
    # The full name of a sample table.
    return quoted(CATALOG, SCHEMA, f'adventureworks_{table}')

## Step 3. Deploy the model

The next cell creates a Model Serving endpoint for the model in your workspace, on one GPU (`GPU_MEDIUM`, size Small) that scales to zero when idle.
It does not wait for the deployment; Step 6 does.
If the endpoint already exists, the cell checks that it serves this model.
You can also create it with **Serve this model** on the model's page in Catalog Explorer.

In [ ]:
workspace = WorkspaceClient()
try:
    endpoint = workspace.serving_endpoints.get(ENDPOINT_NAME)
except NotFound:
    workspace.serving_endpoints.create(
        name=ENDPOINT_NAME,
        config=EndpointCoreConfigInput(
            name=ENDPOINT_NAME,
            served_entities=[
                ServedEntityInput(
                    entity_name=MODEL_NAME,
                    entity_version=MODEL_VERSION,
                    workload_type=ServingModelWorkloadType.GPU_MEDIUM,
                    workload_size='Small',
                    scale_to_zero_enabled=True,
                )
            ],
        ),
    )
    print(
        f'Creating the endpoint {ENDPOINT_NAME}; this takes 20 to 40 minutes.'
    )
else:
    config = endpoint.config or endpoint.pending_config
    served = {
        (entity.entity_name.lower(), entity.entity_version)
        for entity in config.served_entities
    }
    if served != {(MODEL_NAME.lower(), MODEL_VERSION)}:
        raise ValueError(
            f'The endpoint {ENDPOINT_NAME} already exists and serves {sorted(served)}, '
            f'not {MODEL_NAME} version {MODEL_VERSION}. Set ENDPOINT_NAME to another name.'
        )
    print(f'The endpoint {ENDPOINT_NAME} already exists and serves this model.')

## Step 4. Load the sample data

The next cell downloads the AdventureWorks customers, products, order headers and order lines from the [Microsoft SQL Server samples repository](https://github.com/microsoft/sql-server-samples/tree/master/samples/databases/adventure-works/oltp-install-script) at a fixed commit, adds each order's customer and date to its lines, and saves the tables to your schema.
It needs internet access, and skips the download if the tables already exist with the expected columns.

> Each user is responsible for checking the content of datasets and the applicable licenses and determining if suitable for the intended use.
> The repository's samples are provided under the [MIT License](https://github.com/microsoft/sql-server-samples/blob/master/license.txt).

In [ ]:
URL = (
    'https://raw.githubusercontent.com/microsoft/sql-server-samples/'
    'b47eadc85277f72843647223cde53aa1b85ffcb8/samples/databases/'
    'adventure-works/oltp-install-script/{}.csv'
)
# Each table's file, and the file's columns in order.
FILES = {
    'customers': (
        'Customer',
        'CustomerID PersonID StoreID TerritoryID AccountNumber rowguid '
        'ModifiedDate',
    ),
    'products': (
        'Product',
        'ProductID Name ProductNumber MakeFlag FinishedGoodsFlag Color '
        'SafetyStockLevel ReorderPoint StandardCost ListPrice Size '
        'SizeUnitMeasureCode WeightUnitMeasureCode Weight DaysToManufacture '
        'ProductLine Class Style ProductSubcategoryID ProductModelID '
        'SellStartDate SellEndDate DiscontinuedDate rowguid ModifiedDate',
    ),
    'sales_order_headers': (
        'SalesOrderHeader',
        'SalesOrderID RevisionNumber OrderDate DueDate ShipDate Status '
        'OnlineOrderFlag SalesOrderNumber PurchaseOrderNumber AccountNumber '
        'CustomerID SalesPersonID TerritoryID BillToAddressID ShipToAddressID '
        'ShipMethodID CreditCardID CreditCardApprovalCode CurrencyRateID '
        'SubTotal TaxAmt Freight TotalDue Comment rowguid ModifiedDate',
    ),
    'sales_order_details': (
        'SalesOrderDetail',
        'SalesOrderID SalesOrderDetailID CarrierTrackingNumber OrderQty '
        'ProductID SpecialOfferID UnitPrice UnitPriceDiscount LineTotal '
        'rowguid ModifiedDate',
    ),
}
# The columns each saved table has: order lines also get CustomerID and OrderDate.
COLUMNS = {table: columns.split() for table, (_, columns) in FILES.items()}
COLUMNS['sales_order_details'] += ['CustomerID', 'OrderDate']

existing = [
    table for table in FILES if spark.catalog.tableExists(full_name(table))
]
for table in existing:
    if spark.table(full_name(table)).columns != COLUMNS[table]:
        raise ValueError(
            f'{full_name(table)} already exists with other columns. '
            'Drop it, or set SCHEMA to another schema.'
        )
if len(existing) == len(FILES):
    print(f'The sample tables are already in {CATALOG}.{SCHEMA}.')
else:
    frames = {}
    for table, (file, columns) in FILES.items():
        frame = pd.read_csv(
            URL.format(file),
            sep='\t',
            header=None,
            names=columns.split(),
            dtype_backend='numpy_nullable',
        )
        for column in frame.columns:
            if column.endswith('Date'):
                frame[column] = pd.to_datetime(frame[column])
        frames[table] = frame
    orders = frames['sales_order_headers'][
        ['SalesOrderID', 'CustomerID', 'OrderDate']
    ]
    frames['sales_order_details'] = frames['sales_order_details'].merge(
        orders, on='SalesOrderID', validate='many_to_one'
    )
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {quoted(CATALOG, SCHEMA)}')
    for table, frame in frames.items():
        spark.createDataFrame(frame).write.mode('ignore').saveAsTable(
            full_name(table)
        )
    print(f'Saved the sample tables to {CATALOG}.{SCHEMA}.')

## Step 5. Describe the tables

The model needs each table's primary key, how the tables link, and which tables are event tables.
`DatabricksSource` reads the tables in place, with your permissions, and sends the endpoint only the rows each prediction needs.
Order headers and order lines are event tables, with `OrderDate` as their time column; customers and products have none.

The cell also leaves out columns that would mislead the model: random IDs and free text, which the model does not read, and columns that can show what happened after the prediction time, such as edit times, product sell dates and ship dates.

In [ ]:
source = DatabricksSource(spark, catalog=CATALOG, schema=SCHEMA)
schema = Schema.from_source(
    source,
    {
        'customers': 'adventureworks_customers',
        'products': 'adventureworks_products',
        'sales_order_headers': 'adventureworks_sales_order_headers',
        'sales_order_details': 'adventureworks_sales_order_details',
    },
    primary_keys={
        'customers': 'CustomerID',
        'products': 'ProductID',
        'sales_order_headers': 'SalesOrderID',
        'sales_order_details': 'SalesOrderDetailID',
    },
    time_columns={
        'sales_order_headers': 'OrderDate',
        'sales_order_details': 'OrderDate',
        'customers': None,
        'products': None,
    },
    relationships=[
        Relationship(
            'sales_order_headers', 'CustomerID', 'customers', 'CustomerID'
        ),
        Relationship(
            'sales_order_details', 'CustomerID', 'customers', 'CustomerID'
        ),
        Relationship(
            'sales_order_details', 'ProductID', 'products', 'ProductID'
        ),
    ],
)
leave_out = {
    'customers': ['rowguid', 'ModifiedDate', 'AccountNumber'],
    'products': [
        'rowguid',
        'ModifiedDate',
        'Name',
        'ProductNumber',
        'SellStartDate',
        'SellEndDate',
        'DiscontinuedDate',
    ],
    'sales_order_headers': [
        'rowguid',
        'ModifiedDate',
        'SalesOrderNumber',
        'PurchaseOrderNumber',
        'AccountNumber',
        'CreditCardApprovalCode',
        'Comment',
        'RevisionNumber',
        'Status',
        'ShipDate',
    ],
    'sales_order_details': ['rowguid', 'ModifiedDate', 'CarrierTrackingNumber'],
}
for table, columns in leave_out.items():
    for column in columns:
        del schema[table][column]

print(*schema.tables.values(), sep='\n')
print(*schema.relationships, sep='\n')

## Step 6. Connect to the model

The next cell waits until the endpoint is ready, then connects to it.
If the endpoint has scaled to zero, the first prediction takes a few extra minutes while it starts.

In [ ]:
endpoint = workspace.serving_endpoints.wait_get_serving_endpoint_not_updating(
    ENDPOINT_NAME, timeout=timedelta(hours=1)
)
ready = endpoint.state.ready.value if endpoint.state.ready else 'unknown'
if ready != 'READY':
    raise RuntimeError(
        f'The endpoint {ENDPOINT_NAME} is {ready}. '
        'Open it under Serving to see why, then run this cell again.'
    )
# Allow 15 minutes per request, for an endpoint that has to scale up from zero.
client = SDMClient.for_databricks_serving(ENDPOINT_NAME, timeout=900)
model = KumoRelational()
print(f'Connected to {ENDPOINT_NAME}.')

## Step 7. Make predictions

Each prediction takes three calls:

1. `PredictiveQuery(text, schema)` writes the predictive query.
2. `materialize(indices=[...])` reads the rows the model needs for the given entity IDs, context examples included.
3. `client.predict(request, model=model)` sends the request, and `to_pandas()` returns the result.

**Note:** AdventureWorks is sample data, and the results below are for demonstration.
Benchmark Kumo Relational on data representative of your own use case before drawing conclusions about its performance.

#### Predictive queries in short

<details>
<summary><b>Click here for a short introduction to predictive queries</b></summary>

**Entities**

`FOR EACH <table>.<primary key>` names the entity, such as `FOR EACH customers.CustomerID`; you pass the IDs to predict for as `indices`, so one query works for any set of IDs.

**Targets**

- `PREDICT <table>.<column>` predicts a value of the entity itself, such as a missing attribute.
- `PREDICT <aggregation>(<table>.<column>, <start>, <end>, <unit>)` predicts an aggregate over a time window after the prediction time, such as `0, 30, days` for the next 30 days.

**Aggregations**

`COUNT`, `SUM`, `AVG`, `MIN` and `MAX`; `COUNT(<table>.*, ...)` counts rows, such as orders.

**Conditions**

A comparison after the target turns it into a yes-or-no question: `COUNT(sales_order_headers.*, 0, 90, days) = 0` asks whether a customer places no order in the next 90 days.
A `WHERE` inside the aggregation counts only some rows: `COUNT(sales_order_headers.* WHERE sales_order_headers.TotalDue > 1000, 0, 90, days)` counts only orders over $1,000.
</details>

### Example 1: Forecast product demand

How many units of each of the 10 best-selling products of the last 90 days will be ordered in the next 30 days?

```
PREDICT SUM(sales_order_details.OrderQty, 0, 30, days)   ← units ordered in the next 30 days
FOR EACH products.ProductID                              ← one prediction per product
```

`num_context_samples` sets how many context examples the model reads (1,000 by default, up to 10,000).
This notebook uses 2,000 for product questions and 5,000 for customer questions, whose examples are smaller, to stay well under the 16 MB request limit.

In [ ]:
DEMAND = (
    'PREDICT SUM(sales_order_details.OrderQty, 0, 30, days) '
    'FOR EACH products.ProductID'
)
latest = pd.Timestamp(
    spark.sql(
        f'SELECT max(OrderDate) FROM {full_name("sales_order_headers")}'
    ).first()[0]
)
names = (
    spark.table(full_name('products')).toPandas().set_index('ProductID')['Name']
)
best_sellers = spark.sql(f"""
    SELECT ProductID
    FROM {full_name('sales_order_details')}
    WHERE OrderDate > '{latest - timedelta(days=90)}'
    GROUP BY ProductID
    ORDER BY sum(OrderQty) DESC, ProductID
    LIMIT 10
""").toPandas()

request = PredictiveQuery(DEMAND, schema).materialize(
    indices=best_sellers['ProductID'].tolist(), num_context_samples=2000
)
demand = client.predict(request, model=model).to_pandas()

best_sellers['product'] = best_sellers['ProductID'].map(names)
# A forecast for a product that has stopped selling can dip just below zero.
best_sellers['forecast'] = best_sellers['ProductID'].map(
    demand.set_index('id')['prediction'].clip(lower=0)
)
display(best_sellers.round(1))

How to interpret the result:

1. `ProductID` and `product`: the product each forecast is for.
2. `forecast`: the predicted units ordered in the 30 days after the last order in the data.

This result can support inventory and demand planning.
Example 2 shows how to check a forecast like this against actual sales.

### Example 2: Check a forecast against actual sales

How good is the forecast?
To find out, move the prediction time into the past with `anchor_time`.
The model then sees only the orders placed up to that time, so its forecast can be compared with what was ordered afterwards.
Here the prediction time is 180 days before the last order, and the forecast is for the 30 products ordered most up to then.

In [ ]:
PAST = latest - timedelta(days=180)


def actual_sales(column, days):
    # What each product actually sold in the `days` after the prediction time.
    return (
        spark.sql(f"""
            SELECT ProductID, sum({column}) AS total
            FROM {full_name('sales_order_details')}
            WHERE OrderDate > '{PAST}' AND OrderDate <= '{PAST + timedelta(days=days)}'
            GROUP BY ProductID
        """)
        .toPandas()
        .set_index('ProductID')['total']
    )


def median_error(forecast, actual):
    # The typical gap between forecast and actual, as a share of the actual
    # value, over the products that sold anything.
    sold = actual > 0
    error = ((forecast - actual).abs() / actual)[sold].median()
    return f'{error:.0%}' if sold.any() else 'not available'


past_best_sellers = spark.sql(f"""
    SELECT ProductID
    FROM {full_name('sales_order_details')}
    WHERE OrderDate <= '{PAST}'
    GROUP BY ProductID
    ORDER BY sum(OrderQty) DESC, ProductID
    LIMIT 30
""").toPandas()
request = PredictiveQuery(DEMAND, schema).materialize(
    indices=past_best_sellers['ProductID'].tolist(),
    anchor_time=PAST,
    num_context_samples=2000,
)
past_demand = client.predict(request, model=model).to_pandas()

past_best_sellers['product'] = past_best_sellers['ProductID'].map(names)
past_best_sellers['forecast'] = past_best_sellers['ProductID'].map(
    past_demand.set_index('id')['prediction'].clip(lower=0)
)
past_best_sellers['actual'] = (
    past_best_sellers['ProductID'].map(actual_sales('OrderQty', 30)).fillna(0)
)
stopped = past_best_sellers[past_best_sellers['actual'] == 0]
print(f'Prediction time: {PAST.date()}.')
print(
    f'Median forecast error: {median_error(past_best_sellers["forecast"], past_best_sellers["actual"])}.'
)
print(
    f'{len(stopped)} products were not ordered at all; '
    f'their largest forecast was {stopped["forecast"].max():.1f} units.'
)
ax = past_best_sellers.plot.scatter(
    'actual', 'forecast', title='Forecast and actual units, 30 days'
)
ax.axline(
    (0, 0), slope=1, color='gray', linewidth=1
)  # exact forecasts lie on this line
display(
    past_best_sellers[['ProductID', 'product', 'forecast', 'actual']]
    .head(10)
    .round(1)
)

How to interpret the result:

1. `forecast` and `actual`: the predicted and the actual units ordered in the 30 days, for the first 10 products; the chart shows all 30.
2. The printed lines give the median error and the forecasts for the products that were not ordered at all.

Check your own queries this way before you rely on them.

### Example 3: Predict which customers will stop ordering

Which active customers will place no order in the next 90 days?

```
PREDICT COUNT(sales_order_headers.*, 0, 90, days) = 0   ← no orders in the next 90 days
FOR EACH customers.CustomerID                           ← one prediction per customer
```

The `= 0` makes it a yes-or-no question, answered with a probability.
The cell asks it at the prediction time of Example 2, for a random sample of 200 customers who ordered in the year before, and reads what each did in the next 90 days; Examples 4 and 5 use the same customers.

In [ ]:
NEXT_90_DAYS = (
    f"OrderDate > '{PAST}' AND OrderDate <= '{PAST + timedelta(days=90)}'"
)
customers = (
    spark.sql(f"""
    SELECT CustomerID,
           count_if({NEXT_90_DAYS}) AS orders,
           coalesce(sum(TotalDue) FILTER (WHERE {NEXT_90_DAYS}), 0) AS spend,
           count_if({NEXT_90_DAYS} AND TotalDue > 1000) AS large_orders
    FROM {full_name('sales_order_headers')}
    GROUP BY CustomerID
    HAVING count_if(OrderDate <= '{PAST}' AND OrderDate > '{PAST - timedelta(days=365)}') > 0
    ORDER BY xxhash64(CustomerID), CustomerID
    LIMIT 200
""")
    .toPandas()
    .set_index('CustomerID')
)


def by_group(probability, happened):
    # Four equal groups by predicted probability: the average prediction, and
    # how often the event actually happened.
    if len(probability) < 4:
        raise ValueError('Four groups need at least four customers.')
    labels = ['lowest', 'low', 'high', 'highest']
    group = pd.qcut(probability.rank(method='first'), 4, labels=labels).rename(
        'group'
    )
    frame = pd.DataFrame(
        {'predicted': probability, 'actual': happened.astype(float)}
    )
    return frame.groupby(group, observed=True).mean().round(2)


request = PredictiveQuery(
    'PREDICT COUNT(sales_order_headers.*, 0, 90, days) = 0 '
    'FOR EACH customers.CustomerID',
    schema,
).materialize(
    indices=customers.index.tolist(), anchor_time=PAST, num_context_samples=5000
)
churn = (
    client.predict(request, model=model).to_pandas().join(customers, on='id')
)
display(by_group(churn['probability_True'], churn['orders'] == 0))

How to interpret the result:

1. Each row is a quarter of the 200 customers, grouped by `probability_True`, the predicted probability of placing no order.
2. `predicted` and `actual`: the group's average probability, and the share that placed no order.

Groups rated riskier stopped ordering more often, so the probability can rank customers for retention work.

### Example 4: Predict each customer's spend

How much will each of the same customers spend in the next 90 days?

```
PREDICT SUM(sales_order_headers.TotalDue, 0, 90, days)   ← order value in the next 90 days
FOR EACH customers.CustomerID                            ← one prediction per customer
```

`prediction_statistic='mean'` makes each prediction the expected spend rather than the median.
Most customers spend nothing in a given 90 days, so their median is 0, while expected spend adds up across customers.

In [ ]:
request = PredictiveQuery(
    'PREDICT SUM(sales_order_headers.TotalDue, 0, 90, days) '
    'FOR EACH customers.CustomerID',
    schema,
).materialize(
    indices=customers.index.tolist(),
    anchor_time=PAST,
    num_context_samples=5000,
    prediction_statistic='mean',
)
spend = (
    client.predict(request, model=model).to_pandas().join(customers, on='id')
)
spend['predicted'] = spend['prediction'].clip(lower=0)
top_20 = spend.nlargest(20, 'predicted')
if spend['spend'].sum() > 0:
    share = top_20['spend'].sum() / spend['spend'].sum()
    print(
        f'The 20 customers with the highest predicted spend made {share:.0%} of the actual spend.'
    )
display(top_20.head(10)[['id', 'predicted', 'spend']].round(0))

How to interpret the result:

1. `id`, `predicted` and `spend`: the customer, the expected spend in the 90 days, and the actual spend.

A random 20 customers would make about a tenth of the spend, so a high printed share means the prediction finds the customers who matter most.

### Example 5: Predict who will place a large order

Which of the same customers will place an order over $1,000 in the next 90 days?

```
PREDICT COUNT(sales_order_headers.*
              WHERE sales_order_headers.TotalDue > 1000, 0, 90, days) > 0   ← any order over $1,000 in 90 days
FOR EACH customers.CustomerID                                               ← one prediction per customer
```

The `WHERE` inside the aggregation counts only large orders, and `> 0` asks whether there is at least one.

In [ ]:
request = PredictiveQuery(
    'PREDICT COUNT(sales_order_headers.* WHERE sales_order_headers.TotalDue > 1000, '
    '0, 90, days) > 0 FOR EACH customers.CustomerID',
    schema,
).materialize(
    indices=customers.index.tolist(), anchor_time=PAST, num_context_samples=5000
)
large = (
    client.predict(request, model=model).to_pandas().join(customers, on='id')
)
display(by_group(large['probability_True'], large['large_orders'] > 0))

How to interpret the result:

1. Each row is a quarter of the customers, grouped by `probability_True`, the predicted probability of a large order.
2. `predicted` and `actual`: the group's average probability, and the share that placed one.

Large orders came mostly from the group rated most likely, so the probability can pick whom to contact for an upsell.

### Example 6: Fill in a missing product attribute

A query with no time window predicts a value of the entity itself, which fills in values that are missing.

```
PREDICT products.ProductLine   ← R road, M mountain, T touring or S standard
FOR EACH products.ProductID    ← one prediction per product
```

To check it, the cell predicts the product line of 60 random products whose line is known, and compares the predictions with the recorded values.
The model never sees a product's own recorded value.

In [ ]:
lines = spark.sql(f"""
    SELECT ProductID, Name AS product, trim(ProductLine) AS recorded
    FROM {full_name('products')}
    WHERE ProductLine IS NOT NULL
    ORDER BY xxhash64(ProductID), ProductID
    LIMIT 60
""").toPandas()

request = PredictiveQuery(
    'PREDICT products.ProductLine FOR EACH products.ProductID', schema
).materialize(indices=lines['ProductID'].tolist())
product_lines = client.predict(request, model=model).to_pandas()

lines['predicted'] = lines['ProductID'].map(
    product_lines.set_index('id')['prediction'].str.strip()
)
correct = (lines['predicted'] == lines['recorded']).sum()
print(f'{correct} of {len(lines)} product lines predicted correctly.')
display(lines[lines['predicted'] != lines['recorded']])

How to interpret the result:

1. The printed line counts the correct predictions; always guessing the most common line gets about a third right.
2. The table lists the products the model got wrong.

Predicted values like these can fill gaps in a catalog, but review them before writing them back to a source system.

### Example 7: Predict from a metric view

A Unity Catalog metric view defines how tables join and which measures matter, such as revenue.
The client reads that definition, so you can predict a measure without describing the tables yourself.
The next cell creates or replaces a metric view over the sample tables, `adventureworks_sales_metrics`.

In [ ]:
view_name = full_name('sales_metrics')
spark.sql(f"""
CREATE OR REPLACE VIEW {view_name}
WITH METRICS LANGUAGE YAML AS $$
version: 0.1
source: {json.dumps(full_name('sales_order_details'))}
joins:
  - name: customers
    source: {json.dumps(full_name('customers'))}
    on: source.CustomerID = customers.CustomerID
  - name: products
    source: {json.dumps(full_name('products'))}
    on: source.ProductID = products.ProductID
dimensions:
  - name: SalesOrderDetailID
    expr: source.SalesOrderDetailID
  - name: SalesOrderID
    expr: source.SalesOrderID
  - name: OrderDate
    expr: source.OrderDate
  - name: OrderQty
    expr: source.OrderQty
  - name: UnitPrice
    expr: source.UnitPrice
  - name: LineTotal
    expr: source.LineTotal
  - name: Discounted
    expr: source.UnitPriceDiscount > 0
  - name: TerritoryID
    expr: customers.TerritoryID
  - name: StoreID
    expr: customers.StoreID
  - name: Color
    expr: products.Color
  - name: ListPrice
    expr: products.ListPrice
  - name: ProductLine
    expr: products.ProductLine
  - name: ProductSubcategoryID
    expr: products.ProductSubcategoryID
measures:
  - name: total_quantity
    expr: SUM(source.OrderQty)
  - name: revenue
    expr: SUM(source.LineTotal)
$$
""")
print(f'Created the metric view {view_name}.')

`source.semantic_model` reads the view, and `Schema.from_source` turns it into a schema.
The cell prints the predictive-query target behind each measure; the next cell forecasts `revenue` for the 30 products of Example 2 and shows the actual revenue next to it.

In [ ]:
view_schema = Schema.from_source(source.semantic_model(view_name))
print(*view_schema.conversion_notes, sep='\n')

In [ ]:
request = PredictiveQuery(
    'PREDICT SUM(adventureworks_sales_order_details.LineTotal, 0, 30, days) '
    'FOR EACH products.ProductID',
    view_schema,
).materialize(
    indices=past_best_sellers['ProductID'].tolist(),
    anchor_time=PAST,
    num_context_samples=2000,
)
revenue = client.predict(request, model=model).to_pandas()

past_best_sellers['revenue_forecast'] = past_best_sellers['ProductID'].map(
    revenue.set_index('id')['prediction'].clip(lower=0)
)
past_best_sellers['actual_revenue'] = (
    past_best_sellers['ProductID'].map(actual_sales('LineTotal', 30)).fillna(0)
)
error = median_error(
    past_best_sellers['revenue_forecast'], past_best_sellers['actual_revenue']
)
print(f'Median forecast error: {error}.')
display(
    past_best_sellers[
        ['ProductID', 'product', 'revenue_forecast', 'actual_revenue']
    ]
    .head(10)
    .round(0)
)

How to interpret the result:

1. `revenue_forecast` and `actual_revenue`: the predicted and the actual revenue in the 30 days, from the view's `revenue` measure.

This lets a team forecast the measures it already defines in metric views.

## Step 8. Try it yourself

The next cell runs the Example 2 forecast again for its 30 products, and shows the first 10 next to the actual values.
Change `DAYS` or `COLUMN`, or uncomment a challenge line, and run it again.

In [ ]:
DAYS = 30
COLUMN = 'OrderQty'
# Challenge: uncomment a line, run the cell again, and compare with the actuals.
# DAYS = 90  # forecast the next 90 days instead of 30
# COLUMN = 'LineTotal'  # forecast revenue instead of units

request = PredictiveQuery(
    f'PREDICT SUM(sales_order_details.{COLUMN}, 0, {DAYS}, days) '
    'FOR EACH products.ProductID',
    schema,
).materialize(
    indices=past_best_sellers['ProductID'].tolist(),
    anchor_time=PAST,
    num_context_samples=2000,
)
result = client.predict(request, model=model).to_pandas()

result['product'] = result['id'].map(names)
result['prediction'] = result['prediction'].clip(lower=0)
result['actual'] = result['id'].map(actual_sales(COLUMN, DAYS)).fillna(0)
display(result[['id', 'product', 'prediction', 'actual']].head(10).round(1))

## Step 9. Use your own data

For your own tables, describe them and write a predictive query:

```python
source = DatabricksSource(spark, catalog='<your catalog>', schema='<your schema>')
schema = Schema.from_source(
    source,
    {'users': '<users table>', 'orders': '<orders table>'},
    primary_keys={'users': 'user_id', 'orders': 'order_id'},
    time_columns={'orders': 'ordered_at', 'users': None},
    relationships=[Relationship('orders', 'user_id', 'users', 'user_id')],
)
request = PredictiveQuery(
    'PREDICT COUNT(orders.*, 0, 90, days) = 0 FOR EACH users.user_id', schema
).materialize(indices=[...], num_context_samples=5000)
result = client.predict(request, model=model).to_pandas()
```

| Question | Predictive query |
|---|---|
| Will this customer stop ordering? | `PREDICT COUNT(orders.*, 0, 90, days) = 0 FOR EACH users.user_id` |
| How much will this customer spend? | `PREDICT SUM(orders.amount, 0, 90, days) FOR EACH users.user_id` |
| How many units of this item will be ordered? | `PREDICT SUM(order_lines.quantity, 0, 30, days) FOR EACH items.item_id` |
| What is this item's missing category? | `PREDICT items.category FOR EACH items.item_id` |

Tips:

- Leave out random row IDs, free text, and columns filled in after the prediction time, such as edit times and ship dates.
- To keep the predictions, save them as a table: `spark.createDataFrame(result).write.saveAsTable('<table>')`.

Learn more:

- [Predictive Query reference](https://docs.nvidia.com/sdgm/reference/pq-reference-overview)
- [Writing predictive queries](https://docs.nvidia.com/sdgm/rfm/writing-predictive-queries)

## Step 10. Clean up

The endpoint bills for GPU time only while it runs.
To delete it, uncomment the last line and run the cell.
The sample tables and the metric view stay in your schema until you drop them.

In [ ]:
client.close()
# workspace.serving_endpoints.delete(ENDPOINT_NAME)